### 7. Live RAG Development

#### 1. Objetivo

El objetivo de esta fase es desarrollar el componente RAG utilizado por el sistema en modo live.

El componente recibe una claim y los documentos candidatos recuperados previamente por el Research Agent. A partir de estos documentos se realizará la limpieza del contenido, su división en fragmentos, la generación de embeddings y la construcción de un índice vectorial.

Posteriormente, utilizando la claim como consulta, se recuperarán los fragmentos más relevantes que serán utilizados como evidencias de entrada para el Evidence Verifier.

A diferencia del benchmark AVeriTeC, este componente debe ser capaz de procesar contenido tanto en español como en inglés, por lo que se utilizará un modelo de `embeddings multilingüe`.

### 2. Modelo de embeddings multilingüe

A diferencia del benchmark AVeriTeC, el sistema en modo live puede recuperar documentos tanto en español como en inglés.

Por este motivo se utiliza el modelo `intfloat/multilingual-e5-small`, diseñado para recuperación semántica multilingüe.

El modelo genera representaciones vectoriales de 384 dimensiones y permite comparar semánticamente una claim con fragmentos documentales escritos en distintos idiomas.

Siguiendo la metodología del modelo E5, las consultas y los documentos se representarán utilizando respectivamente los prefijos `query:` y `passage:`.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient

load_dotenv("../.env")

client = OpenAI()
tavily_client = TavilyClient()

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [49]:
from src.live_retrieval import load_embedding_model
from src.live_retrieval import prepare_live_documents
from src.live_retrieval import chunk_live_documents
from src.live_retrieval import embed_chunks
from src.live_retrieval import build_faiss_index
from src.live_retrieval import retrieve_live_evidence
from src.live_retrieval import run_live_retrieval



from src.research_agent import run_research_agent
from src.evidence_verifier import verify_evidence

In [ ]:
embedding_model = load_embedding_model()

embedding_model

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1651.96it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [8]:
#Pequeña comprobación
test_embedding = embedding_model.encode(
    "query: La Unión Europea limitará los pagos en efectivo en 2027",
    normalize_embeddings=True,
)

test_embedding.shape

(384,)

### 3. Preparación de documentos

Los documentos recuperados por el Research Agent pueden contener ruido derivado del proceso de extracción web, como saltos de línea o espacios repetidos.

Antes de realizar el chunking se aplica una limpieza básica y conservadora del contenido textual, manteniendo los metadatos necesarios para conservar la trazabilidad de cada fuente.

In [11]:
claim = "La Unión Europea prohibirá completamente los pagos en efectivo a partir de 2027."

entities = ["Unión Europea"]

date_reference = "2027"

In [12]:
candidate_documents = run_research_agent(
    claim=claim,
    entities=entities,
    date_reference=date_reference,
    client=client,
    tavily_client=tavily_client,
)

In [13]:
prepared_documents = prepare_live_documents(
    candidate_documents
)

len(prepared_documents)

6

In [ ]:
#Apenas viendo uno de los resultados obtenidos. 
prepared_documents[0]

{'title': 'La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico',
 'url': 'https://www.eldebate.com/economia/consumo/20260820/union-europea-limitara-pagos-efectivo-partir-2027-cantidad-no-podras-pagar-metalico-cns_450803.html',
 'text': 'Fundado en 1910Cerrar sesión ![El BCE aconseja tener dinero en efectivo guardado para posibles emergencias](https://imagenes.eldebate.com/files/new_main_image/uploads/2026/03/17/69b94d637da1e.jpeg) El BCE aconseja tener dinero en efectivo guardado para posibles emergenciasFREEPIK | NICKEL # La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico ## España tiene un límite más restrictivo, por lo que no afectará a los pagos realizados dentro de nuestro país * ## [\u200bIngresos en efectivo: cuándo hay que declararlos en la declaración de la Renta](https://www.eldebate.com/economia/20260321/ingresos-efectivo-cuando-hay-declararlos-declaracion-ren

In [14]:
for document in prepared_documents:
    print("TITLE:", document["title"])
    print("URL:", document["url"])
    print("TEXT LENGTH:", len(document["text"]))
    print("SOURCE TYPE:", document["source_type"])
    print("-" * 80)

TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
URL: https://www.eldebate.com/economia/consumo/20260820/union-europea-limitara-pagos-efectivo-partir-2027-cantidad-no-podras-pagar-metalico-cns_450803.html
TEXT LENGTH: 9864
SOURCE TYPE: web
--------------------------------------------------------------------------------
TITLE: EU Anti-Money Laundering Regulation Effective July 2027
URL: https://www.taylorwessing.com/en/insights-and-events/insights/2026/06/eu-anti-money-laundering-regulation-effective-july-2027
TEXT LENGTH: 6484
SOURCE TYPE: web
--------------------------------------------------------------------------------
TITLE: Payment by Cash or Card? Restrictions on Cash Circulation in ...
URL: https://link.springer.com/article/10.1007/s11196-025-10374-w
TEXT LENGTH: 82552
SOURCE TYPE: web
--------------------------------------------------------------------------------
TITLE: La Unión Europea establece un nuevo lím

### 4. Chunking de documentos

Los documentos recuperados pueden contener varios miles de caracteres y abordar diferentes aspectos del tema analizado.

Para realizar posteriormente una recuperación semántica precisa, cada documento se divide en fragmentos de menor tamaño.

Se utiliza un tamaño máximo de 1000 caracteres con un solapamiento de 150 caracteres entre fragmentos consecutivos. El solapamiento permite conservar parte del contexto cuando una información relevante se encuentra próxima al límite entre dos chunks.

Cada fragmento mantiene los metadatos del documento original, permitiendo conservar la trazabilidad de la evidencia recuperada.

#### 4.1. Prueba del chunking

Se aplica el proceso de fragmentación a los documentos preparados con el objetivo de comprobar cuántos chunks se generan y verificar que cada fragmento conserva los metadatos de la fuente original.

In [18]:
chunks = chunk_live_documents(
    prepared_documents
)

len(chunks)

170

In [19]:
for chunk in chunks[:5]:
    print("TITLE:", chunk["title"])
    print("CHUNK INDEX:", chunk["chunk_index"])
    print("TEXT LENGTH:", len(chunk["text"]))
    print("TEXT:", chunk["text"][:300], "...")
    print("-" * 80)

TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
CHUNK INDEX: 0
TEXT LENGTH: 999
TEXT: Fundado en 1910Cerrar sesión ![El BCE aconseja tener dinero en efectivo guardado para posibles emergencias](https://imagenes.eldebate.com/files/new_main_image/uploads/2026/03/17/69b94d637da1e.jpeg) El BCE aconseja tener dinero en efectivo guardado para posibles emergenciasFREEPIK | NICKEL # La Unión ...
--------------------------------------------------------------------------------
TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
CHUNK INDEX: 1
TEXT LENGTH: 998
TEXT: [El Debate](/autor/redaccion-el-debate/) En España, la Agencia Tributaria presta mucha atención a los **pagos en efectivo**. Tal y como establece la Ley 11/2021, de medidas de prevención y lucha contra el fraude fiscal, en nuestro país no pueden realizarse transacciones comerciales de más de **1.000 ...

In [20]:
from collections import Counter

chunks_per_document = Counter(
    chunk["title"] for chunk in chunks
)

for title, count in chunks_per_document.items():
    print(count, "-", title)

12 - La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
8 - EU Anti-Money Laundering Regulation Effective July 2027
99 - Payment by Cash or Card? Restrictions on Cash Circulation in ...
9 - La Unión Europea establece un nuevo límite para pagos ...
23 - Europa limitará a 10.000 euros los pagos en efectivo, pero ...
19 - AMLR: What changes in 2027?


#### 4.1.1. Análisis del resultado y ajuste de la limpieza

La primera ejecución del proceso de chunking confirma que los documentos se dividen correctamente en fragmentos de aproximadamente 1000 caracteres, manteniendo el solapamiento definido y los metadatos de la fuente original.

Sin embargo, la inspección de los primeros fragmentos muestra que parte del contenido recuperado desde las páginas web contiene elementos que no aportan información útil para la verificación de la claim. Entre ellos aparecen principalmente referencias a imágenes en formato Markdown, enlaces incrustados y contenido propio de la navegación o estructura de las páginas.

Por ejemplo, algunos fragmentos incluyen expresiones del tipo `![texto](url)` o elementos de navegación que podrían introducir ruido en la representación semántica de los documentos.

Antes de generar los embeddings se decide aplicar una limpieza adicional, manteniendo un enfoque conservador para evitar eliminar información relevante. En concreto:

- se eliminan las referencias a imágenes en formato Markdown;
- se conserva el texto visible de los enlaces, eliminando únicamente la URL asociada;
- se mantienen el resto del contenido y los metadatos originales;
- se continúa normalizando los espacios y saltos de línea.

No se aplican reglas específicas para eliminar secciones como recomendaciones, publicidad o bloques de navegación, ya que una limpieza demasiado agresiva podría eliminar contenido válido. Posteriormente, el componente de recuperación semántica deberá priorizar los fragmentos más relacionados con la claim.

Tras modificar la función `clean_document_text()`, es necesario volver a ejecutar la preparación de los documentos y el proceso de chunking para que los fragmentos se generen a partir del texto ya actualizado.

In [23]:
prepared_documents = prepare_live_documents(
    candidate_documents
)

chunks = chunk_live_documents(
    prepared_documents
)

print("Prepared documents:", len(prepared_documents))
print("Chunks:", len(chunks))

Prepared documents: 6
Chunks: 111


In [24]:
for chunk in chunks[:5]:
    print("TITLE:", chunk["title"])
    print("CHUNK INDEX:", chunk["chunk_index"])
    print("TEXT LENGTH:", len(chunk["text"]))
    print("TEXT:", chunk["text"][:300], "...")
    print("-" * 80)

TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
CHUNK INDEX: 0
TEXT LENGTH: 1000
TEXT: Fundado en 1910Cerrar sesión El BCE aconseja tener dinero en efectivo guardado para posibles emergenciasFREEPIK | NICKEL # La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico ## España tiene un límite más restrictivo, por lo que no afectará a ...
--------------------------------------------------------------------------------
TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
CHUNK INDEX: 1
TEXT LENGTH: 999
TEXT: la normativa europea no contempla ningún límite a la hora de realizar pagos en efectivo en sus Estados miembros. Así, como en España, cada nación tiene sus propias reglas. Pero todo cambiará el **10 de julio de 2027**. ## Límite de los pagos en efectivo en la Unión Europea Tal y como establece el Re ..

#### 4.1.2. Resultado tras el ajuste de limpieza

Tras ampliar la limpieza del contenido web, se vuelve a ejecutar la preparación y fragmentación de los documentos.

El número total de chunks se reduce de 170 a 111. Esta disminución se debe principalmente a la eliminación de referencias a imágenes y URLs incrustadas en formato Markdown, que anteriormente ocupaban parte del contenido procesado.

La inspección de los nuevos fragmentos confirma que el texto principal de los documentos se conserva correctamente. Sin embargo, todavía aparecen algunos elementos propios de la estructura de las páginas web, como recomendaciones de otros artículos o secciones de navegación.

No se aplican reglas adicionales específicas para eliminar estos elementos, ya que su estructura varía entre dominios y una limpieza excesivamente agresiva podría eliminar información relevante.

Estos fragmentos se mantienen y será el mecanismo de recuperación semántica del RAG el encargado de priorizar aquellos cuyo contenido sea más similar a la claim analizada.

Con esta configuración se considera finalizada la preparación y fragmentación de los documentos.

### 5. Generación de embeddings de los chunks

Una vez preparados y fragmentados los documentos, cada chunk se transforma en una representación vectorial utilizando el modelo multilingüe `intfloat/multilingual-e5-small`.

Siguiendo la metodología del modelo E5, los fragmentos documentales se codifican utilizando el prefijo `passage:`.

Los embeddings se normalizan durante su generación para poder utilizar posteriormente similitud basada en producto interno, equivalente a similitud coseno cuando los vectores tienen norma unitaria.

In [28]:
chunk_embeddings = embed_chunks(
    chunks=chunks,
    embedding_model=embedding_model,
)

chunk_embeddings.shape

(111, 384)

In [29]:
print("Chunks:", len(chunks))
print("Embeddings:", len(chunk_embeddings))
print("Embedding dimension:", chunk_embeddings.shape[1])

Chunks: 111
Embeddings: 111
Embedding dimension: 384


### 6. Construcción del índice vectorial

Los embeddings generados para los chunks se almacenan en un índice FAISS.

Dado que los vectores han sido previamente normalizados, se utiliza `IndexFlatIP`, que permite ordenar los fragmentos según el producto interno entre la representación de la claim y las representaciones de los documentos.

Con vectores normalizados, este producto interno es equivalente a la similitud coseno.

In [32]:
faiss_index = build_faiss_index(
    chunk_embeddings
)

faiss_index.ntotal

111

In [35]:
print("Chunks:", len(chunks))
print("Vectors in FAISS:", faiss_index.ntotal)

Chunks: 111
Vectors in FAISS: 111


### 7. Recuperación de evidencias

Una vez construido el índice vectorial, la claim se transforma en un embedding utilizando el prefijo `query:` requerido por el modelo E5.

El embedding de la claim se compara con los embeddings de los chunks almacenados en FAISS y se recuperan los fragmentos con mayor similitud semántica.

En esta primera configuración se recuperan los 10 chunks con mayor similitud.

In [38]:
retrieved_evidence = retrieve_live_evidence(
    claim=claim,
    chunks=chunks,
    faiss_index=faiss_index,
    embedding_model=embedding_model,
    top_k=10,
)

len(retrieved_evidence)

10

In [39]:
for evidence in retrieved_evidence:
    print("SCORE:", round(evidence["score"], 4))
    print("TITLE:", evidence["title"])
    print("CHUNK INDEX:", evidence["chunk_index"])
    print("TEXT:", evidence["text"][:400], "...")
    print("-" * 80)

SCORE: 0.9175
TITLE: La Unión Europea establece un nuevo límite para pagos ...
CHUNK INDEX: 0
TEXT: ## La Información económica Volver a 20MINUTOS.ES La UE no ve que Rusia e Israel provocaran la crisis pero sí intentos posteriores de "sacar partido" fútbol Casadó al Dépor, Jonathan David al Atleti y Julián Álvarez se queda: así cerró el mercado de fichajes # La Unión Europea establece un nuevo límite para pagos en efectivo: así afectará a las compras en 2027 Dinero en efectivo.Pixabay Escucha es ...
--------------------------------------------------------------------------------
SCORE: 0.9151
TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
CHUNK INDEX: 0
TEXT: Fundado en 1910Cerrar sesión El BCE aconseja tener dinero en efectivo guardado para posibles emergenciasFREEPIK | NICKEL # La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico ## España tiene un límite más

#### 7.1. Análisis de la primera recuperación

La primera búsqueda semántica recupera correctamente fragmentos relacionados con la claim analizada. Entre los resultados aparecen referencias directas al límite de 10.000 euros, la aplicación de la medida en 2027 y el alcance de la normativa europea.

Sin embargo, se observa una concentración elevada de resultados procedentes de las mismas fuentes. Varios de los diez primeros fragmentos pertenecen al mismo documento, lo que reduce la diversidad de evidencias disponibles para el Evidence Verifier.

Aunque este comportamiento es coherente con una búsqueda basada exclusivamente en similitud semántica, en un sistema de verificación factual resulta conveniente incorporar evidencias procedentes de distintas fuentes.

Por este motivo, se aplicará una restricción de diversidad documental, limitando el número máximo de chunks recuperados por documento.

In [42]:
retrieved_evidence_2 = retrieve_live_evidence(
    claim=claim,
    chunks=chunks,
    faiss_index=faiss_index,
    embedding_model=embedding_model,
    top_k=10,
)

In [43]:
for evidence in retrieved_evidence_2:
    print("SCORE:", round(evidence["score"], 4))
    print("TITLE:", evidence["title"])
    print("CHUNK INDEX:", evidence["chunk_index"])
    print("TEXT:", evidence["text"][:400], "...")
    print("-" * 80)

SCORE: 0.9175
TITLE: La Unión Europea establece un nuevo límite para pagos ...
CHUNK INDEX: 0
TEXT: ## La Información económica Volver a 20MINUTOS.ES La UE no ve que Rusia e Israel provocaran la crisis pero sí intentos posteriores de "sacar partido" fútbol Casadó al Dépor, Jonathan David al Atleti y Julián Álvarez se queda: así cerró el mercado de fichajes # La Unión Europea establece un nuevo límite para pagos en efectivo: así afectará a las compras en 2027 Dinero en efectivo.Pixabay Escucha es ...
--------------------------------------------------------------------------------
SCORE: 0.9151
TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
CHUNK INDEX: 0
TEXT: Fundado en 1910Cerrar sesión El BCE aconseja tener dinero en efectivo guardado para posibles emergenciasFREEPIK | NICKEL # La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico ## España tiene un límite más

#### 7.2. Resultado tras aplicar diversidad documental

Tras incorporar una restricción de máximo dos chunks por documento, la recuperación presenta una mayor diversidad de fuentes.

Los diez fragmentos recuperados proceden ahora de seis documentos diferentes, evitando que una única página concentre la mayor parte de las evidencias proporcionadas al Evidence Verifier.

La recuperación sigue manteniendo fragmentos con alta similitud semántica respecto a la claim, incluyendo información sobre el límite de 10.000 euros, la entrada en aplicación de la normativa en 2027 y referencias al Reglamento (UE) 2024/1624.

La incorporación de diversidad documental implica aceptar algunos fragmentos con una similitud ligeramente inferior a los primeros resultados del ranking original, pero permite disponer de evidencia procedente de distintas fuentes, lo que resulta especialmente relevante en un sistema de verificación factual.

Con esta configuración se considera adecuada la estrategia de recuperación de evidencias del RAG.

### 8. Ejecución completa del Live RAG

Una vez validadas individualmente las distintas etapas del pipeline de recuperación, se integran en una única función de nivel superior, `run_live_retrieval()`.

Esta función recibe la claim, los documentos candidatos obtenidos por el Research Agent y el modelo de embeddings. Internamente realiza la preparación de documentos, el chunking, la generación de embeddings, la construcción del índice FAISS y la recuperación final de evidencias.

La salida corresponde a los fragmentos más relevantes que serán utilizados posteriormente por el Evidence Verifier.

In [47]:
live_evidence = run_live_retrieval(
    claim=claim,
    candidate_documents=candidate_documents,
    embedding_model=embedding_model,
)

len(live_evidence)

10

In [48]:
for evidence in live_evidence:
    print("SCORE:", round(evidence["score"], 4))
    print("TITLE:", evidence["title"])
    print("CHUNK INDEX:", evidence["chunk_index"])
    print("TEXT:", evidence["text"][:400], "...")
    print("-" * 80)

SCORE: 0.9175
TITLE: La Unión Europea establece un nuevo límite para pagos ...
CHUNK INDEX: 0
TEXT: ## La Información económica Volver a 20MINUTOS.ES La UE no ve que Rusia e Israel provocaran la crisis pero sí intentos posteriores de "sacar partido" fútbol Casadó al Dépor, Jonathan David al Atleti y Julián Álvarez se queda: así cerró el mercado de fichajes # La Unión Europea establece un nuevo límite para pagos en efectivo: así afectará a las compras en 2027 Dinero en efectivo.Pixabay Escucha es ...
--------------------------------------------------------------------------------
SCORE: 0.9151
TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
CHUNK INDEX: 0
TEXT: Fundado en 1910Cerrar sesión El BCE aconseja tener dinero en efectivo guardado para posibles emergenciasFREEPIK | NICKEL # La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico ## España tiene un límite más

#### 8.1. Resultado de la ejecución completa

La ejecución completa del pipeline Live RAG recupera diez fragmentos relevantes para la claim analizada.

Los resultados mantienen la diversidad documental introducida previamente, con un máximo de dos chunks por fuente, e incluyen información directamente relacionada con el límite de pagos en efectivo, la aplicación de la normativa a partir de 2027 y referencias al Reglamento (UE) 2024/1624.

La prueba confirma que la función `run_live_retrieval()` integra correctamente las etapas de preparación de documentos, chunking, generación de embeddings, construcción del índice FAISS y recuperación semántica de evidencias.

Con esta configuración se considera validado el componente Live RAG.

### 9. Integración con el Evidence Verifier

Una vez recuperados los fragmentos más relevantes mediante el Live RAG, estos se utilizan como contexto factual para el componente `Evidence Verifier`.

El Evidence Verifier recibe dos elementos principales:

- la claim que debe verificarse;
- las evidencias recuperadas por el RAG.

A partir de estas evidencias, el componente analiza individualmente la relación de cada fragmento con la claim y genera un veredicto global.

De esta forma, el flujo de verificación factual queda conectado de la siguiente manera:

Claim → Research Agent → Live RAG → Evidence Verifier

In [50]:
verification_result = verify_evidence(
    claim=claim,
    evidences=live_evidence,
    client=client,
)

verification_result

VerificationResult(verdict='REFUTED', evidence_sufficient=True, evidence_assessments=[EvidenceAssessment(evidence_id=1, relation='REFUTES', reason='It describes the measure as a new limit on cash payments applying before summer 2027, rather than a complete prohibition of cash payments.'), EvidenceAssessment(evidence_id=2, relation='REFUTES', reason='It explicitly states that the EU will limit cash payments from 2027 and notes that Spain already has a lower cash-payment limit, which is inconsistent with a total ban.'), EvidenceAssessment(evidence_id=3, relation='NEUTRAL', reason='It provides context about cash use and anti-money-laundering motivations but does not state the specific rule or establish whether cash payments will be fully banned.'), EvidenceAssessment(evidence_id=4, relation='REFUTES', reason='It states that the EU measure is a EUR 10,000 maximum and that Spain will retain a EUR 1,000 limit for relevant transactions. A maximum amount is not a complete cash-payment ban.'), 

In [51]:
print("VERDICT:", verification_result.verdict)
print("EVIDENCE SUFFICIENT:", verification_result.evidence_sufficient)
print("EXPLANATION:", verification_result.explanation)
print("-" * 80)

for assessment in verification_result.evidence_assessments:
    print("EVIDENCE ID:", assessment.evidence_id)
    print("RELATION:", assessment.relation)
    print("REASON:", assessment.reason)
    print("-" * 80)

VERDICT: REFUTED
EVIDENCE SUFFICIENT: True
EXPLANATION: The claim is false. The evidence consistently describes an EU-wide ceiling of EUR 10,000 for cash payments for goods and services from 10 July 2027, not a complete ban on using cash. Cash payments at or below the applicable limit remain possible under the described rule.
--------------------------------------------------------------------------------
EVIDENCE ID: 1
RELATION: REFUTES
REASON: It describes the measure as a new limit on cash payments applying before summer 2027, rather than a complete prohibition of cash payments.
--------------------------------------------------------------------------------
EVIDENCE ID: 2
RELATION: REFUTES
REASON: It explicitly states that the EU will limit cash payments from 2027 and notes that Spain already has a lower cash-payment limit, which is inconsistent with a total ban.
--------------------------------------------------------------------------------
EVIDENCE ID: 3
RELATION: NEUTRAL
REASON

In [55]:
print("VERDICT:", verification_result.verdict)
print("EVIDENCE SUFFICIENT:", verification_result.evidence_sufficient)
print("EXPLANATION:", verification_result.explanation)
print("=" * 100)

for assessment in verification_result.evidence_assessments:

    # evidence = live_evidence[assessment.evidence_id]
    evidence = live_evidence[assessment.evidence_id - 1]

    print("EVIDENCE ID:", assessment.evidence_id)
    print("TITLE:", evidence["title"])
    print("URL:", evidence["url"])
    print("RELATION:", assessment.relation)
    print("REASON:", assessment.reason)
    print("TEXT:", evidence["text"][:500], "...")
    print("=" * 100)

VERDICT: REFUTED
EVIDENCE SUFFICIENT: True
EXPLANATION: The claim is false. The evidence consistently describes an EU-wide ceiling of EUR 10,000 for cash payments for goods and services from 10 July 2027, not a complete ban on using cash. Cash payments at or below the applicable limit remain possible under the described rule.
EVIDENCE ID: 1
TITLE: La Unión Europea establece un nuevo límite para pagos ...
URL: https://www.20minutos.es/lainformacion/economia-y-finanzas/union-europea-establece-nuevo-limite-pagos-efectivo-afectara-compras-2027_6935143_0.html
RELATION: REFUTES
REASON: It describes the measure as a new limit on cash payments applying before summer 2027, rather than a complete prohibition of cash payments.
TEXT: ## La Información económica Volver a 20MINUTOS.ES La UE no ve que Rusia e Israel provocaran la crisis pero sí intentos posteriores de "sacar partido" fútbol Casadó al Dépor, Jonathan David al Atleti y Julián Álvarez se queda: así cerró el mercado de fichajes # La Unió

#### 9.1. Resultado de la integración con el Evidence Verifier

El Evidence Verifier clasifica la claim como `REFUTED` y considera que la
evidencia recuperada es suficiente (`evidence_sufficient=True`).

La explicación generada señala que las fuentes recuperadas describen un límite
máximo de 10.000 euros para determinados pagos en efectivo a partir del 10 de
julio de 2027, y no una prohibición completa del uso de efectivo.

El análisis individual de las evidencias muestra dos tipos principales de
relación:

- `REFUTES`: fragmentos que describen explícitamente el límite de 10.000 euros
  y, por tanto, contradicen la afirmación de una prohibición total.
- `NEUTRAL`: fragmentos relacionados con el Reglamento o su fecha de entrada en
  aplicación, pero que no contienen información suficiente sobre la regulación
  concreta de los pagos en efectivo.

No se identifican evidencias clasificadas como `SUPPORTS`.

El resultado confirma que el Live RAG y el Evidence Verifier pueden utilizarse
de forma conjunta: el primero recupera los fragmentos relevantes y el segundo
interpreta su relación factual con la claim antes de producir el veredicto.